In [43]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
import os
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# 1. Загрузка данных

In [2]:

DATA_PATH = '../data/raw/nigerian_mobile_money_full.parquet'
print(f"Загрузка данных из {DATA_PATH}...")
df = pd.read_parquet(DATA_PATH)

Загрузка данных из ../data/raw/nigerian_mobile_money_full.parquet...


In [3]:
# Преобразуем timestamp в datetime и сортируем для временных признаков
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.sort_values(['wallet_id', 'timestamp'], inplace=True)
df.reset_index(drop=True, inplace=True)


In [4]:
# Целевые переменные (оставляем для оценки, но не используем в генерации признаков)
y_fraud = df['fraud_flag'].astype(int)
y_churn = df['churn_30d'].astype(int)

# 2. Базовые признаки

## 2.1 Числовые признаки (нормализация)

In [5]:
numeric_cols = ['amount_ngn', 'fee_ngn', 'balance_after_ngn']
scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

In [6]:
# Дополнительные числовые признаки с экономическим смыслом
df['fee_rate'] = df['fee_ngn'] / (df['amount_ngn'] + 1e-6)  # эффективная комиссия
df['balance_to_amount'] = df['balance_after_ngn'] / (df['amount_ngn'] + 1e-6)
df['is_round_amount'] = df['amount_ngn'].apply(lambda x: x in [50000, 100000, 200000]).astype(int)
df['is_small_amount'] = (df['amount_ngn'] < 1000).astype(int)

## 2.2 Категориальные признаки с низкой кардинальностью

In [7]:
cat_low_cols = ['transaction_type', 'channel', 'device_os', 'kyc_tier']
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded = encoder.fit_transform(df[cat_low_cols])
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(cat_low_cols))
df = pd.concat([df, encoded_df], axis=1)

## 2.3 Высококардинальные идентификаторы (wallet_id, agent_id)

In [8]:
#     Заменяем их статистиками, чтобы избежать перекодирования в one-hot

wallet_stats = df.groupby('wallet_id').agg({
    'amount_ngn': ['count', 'sum', 'mean', 'std'],
    'fee_ngn': 'mean',
    'transaction_type': lambda x: x.nunique(),
    'channel': lambda x: x.nunique()
}).reset_index()
wallet_stats.columns = ['wallet_id', 'wallet_txn_count', 'wallet_total_amount', 
                        'wallet_avg_amount', 'wallet_std_amount', 'wallet_avg_fee',
                        'wallet_type_diversity', 'wallet_channel_diversity']
df = df.merge(wallet_stats, on='wallet_id', how='left')

In [9]:
# Аналогично для агентов (только для cashin/cashout)
agent_stats = df[df['agent_id'].notna()].groupby('agent_id').agg({
    'amount_ngn': ['count', 'sum', 'mean'],
    'transaction_type': lambda x: (x == 'cashout').sum() / len(x)  # доля cashout
}).reset_index()
agent_stats.columns = ['agent_id', 'agent_txn_count', 'agent_total_amount', 
                       'agent_avg_amount', 'agent_cashout_ratio']
df = df.merge(agent_stats, on='agent_id', how='left')
df['agent_txn_count'].fillna(0, inplace=True)
df['agent_total_amount'].fillna(0, inplace=True)
df['agent_avg_amount'].fillna(0, inplace=True)
df['agent_cashout_ratio'].fillna(0, inplace=True)

C:\Users\Admin\AppData\Local\Temp\ipykernel_22236\959767539.py:9: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['agent_txn_count'].fillna(0, inplace=True)
C:\Users\Admin\AppData\Local\Temp\ipykernel_22236\959767539.py:10: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment usi

0          0.000000
1          0.465116
2          0.000000
3          0.000000
4          0.000000
             ...   
3999995    0.000000
3999996    0.000000
3999997    0.000000
3999998    0.000000
3999999    0.486111
Name: agent_cashout_ratio, Length: 4000000, dtype: float64

# 3. Временные признаки (скользящие окна по кошельку)

In [10]:
# Сортировка по времени уже выполнена
df.set_index('timestamp', inplace=True)

In [11]:
# Функция для вычисления скользящих агрегатов
def rolling_features(group, windows):
    group = group.sort_index()
    for window in windows:
        # Количество транзакций
        group[f'txn_count_{window}'] = group['amount_ngn'].rolling(window, closed='left').count()
        # Сумма
        group[f'amount_sum_{window}'] = group['amount_ngn'].rolling(window, closed='left').sum()
        # Средняя сумма
        group[f'amount_mean_{window}'] = group['amount_ngn'].rolling(window, closed='left').mean()
        # Стандартное отклонение (заполняем NaN нулём)
        group[f'amount_std_{window}'] = group['amount_ngn'].rolling(window, closed='left').std().fillna(0)
    return group

In [12]:
tqdm.pandas()

In [13]:
# Вычисляем для окон: 1 час, 24 часа, 7 дней
windows = ['1h', '24h', '7D']
df = df.groupby('wallet_id', group_keys=False).progress_apply(lambda x: rolling_features(x, windows))


100%|██████████| 375837/375837 [35:13<00:00, 177.81it/s]  


In [14]:
wallets_df =pd.read_parquet(DATA_PATH)[['transaction_id','wallet_id','timestamp']]

In [15]:
df = df.merge(wallets_df, on='transaction_id', how='left')

In [16]:
# Время с последней транзакции (в секундах)
df['time_since_last'] = df.groupby('wallet_id')['timestamp'].diff().dt.total_seconds().fillna(0)

In [17]:
# Время с последней транзакции того же типа (упрощённо: разница с предыдущей записью того же типа)
df['time_since_last_same_type'] = df.groupby(['wallet_id', 'transaction_type'])['timestamp'].diff().dt.total_seconds().fillna(0)

In [18]:
# Дополнительные временные признаки: час, день недели, выходной, ночь

df['hour'] = df['timestamp'].apply(lambda x: x.hour)
df['dayofweek'] = df['timestamp'].apply(lambda x: x.dayofweek)
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
df['is_night'] = ((df['hour'] >= 22) | (df['hour'] <= 6)).astype(int)

# 4. Графовые признаки (взаимодействия кошелёк-агент)

In [19]:
# Статистики по парам (wallet_id, agent_id) – только для cashin/cashout
wallet_agent = df[df['agent_id'].notna()].groupby(['wallet_id', 'agent_id']).agg({
    'amount_ngn': ['count', 'sum', 'mean']
}).reset_index()
wallet_agent.columns = ['wallet_id', 'agent_id', 'pair_txn_count', 'pair_total_amount', 'pair_avg_amount']
df = df.merge(wallet_agent, on=['wallet_id', 'agent_id'], how='left')
df['pair_txn_count'].fillna(0, inplace=True)
df['pair_total_amount'].fillna(0, inplace=True)
df['pair_avg_amount'].fillna(0, inplace=True)

# Количество уникальных агентов, с которыми взаимодействовал кошелёк
unique_agents = df[df['agent_id'].notna()].groupby('wallet_id')['agent_id'].nunique().reset_index()
unique_agents.columns = ['wallet_id', 'unique_agents_count']
df = df.merge(unique_agents, on='wallet_id', how='left')
df['unique_agents_count'].fillna(0, inplace=True)

C:\Users\Admin\AppData\Local\Temp\ipykernel_22236\980163571.py:7: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['pair_txn_count'].fillna(0, inplace=True)
C:\Users\Admin\AppData\Local\Temp\ipykernel_22236\980163571.py:8: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using

0          8397
1          8397
2          8397
3          8397
4          8397
           ... 
3999995       1
3999996       1
3999997       1
3999998       2
3999999       2
Name: unique_agents_count, Length: 4000000, dtype: int64

# 5. Обучение без учителя: кластеризация и детекция аномалий

In [20]:
feature_cols = (numeric_cols + 
                ['fee_rate', 'balance_to_amount', 'is_round_amount', 'is_small_amount'] +
                list(encoded_df.columns) +
                ['wallet_txn_count', 'wallet_total_amount', 'wallet_avg_amount', 'wallet_std_amount',
                 'wallet_avg_fee', 'wallet_type_diversity', 'wallet_channel_diversity',
                 'agent_txn_count', 'agent_total_amount', 'agent_avg_amount', 'agent_cashout_ratio',
                 'txn_count_1h', 'amount_sum_1h', 'amount_mean_1h', 'amount_std_1h',
                 'txn_count_24h', 'amount_sum_24h', 'amount_mean_24h', 'amount_std_24h',
                 'txn_count_7D', 'amount_sum_7D', 'amount_mean_7D', 'amount_std_7D',
                 'time_since_last', 'time_since_last_same_type',
                 'hour', 'dayofweek', 'is_weekend', 'is_night',
                 'pair_txn_count', 'pair_total_amount', 'pair_avg_amount',
                 'unique_agents_count'])

X_pre = df[feature_cols].fillna(0).values

n_total = len(df)
sample_size = min(100000, int(0.1 * n_total))
if n_total > sample_size:
    sample_idx = np.random.choice(n_total, size=sample_size, replace=False)
    X_sample = X_pre[sample_idx]
else:
    X_sample = X_pre

scaler_unsup = StandardScaler()
X_sample_scaled = scaler_unsup.fit_transform(X_sample)
X_full_scaled = scaler_unsup.transform(X_pre)

n_clusters = 10
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init='auto')
kmeans.fit(X_sample_scaled)
df['cluster_label'] = kmeans.predict(X_full_scaled)
distances = kmeans.transform(X_full_scaled)
for i in range(n_clusters):
    df[f'cluster_{i}_dist'] = distances[:, i]
cluster_onehot = pd.get_dummies(df['cluster_label'], prefix='cluster')
df = pd.concat([df, cluster_onehot], axis=1)

iso_forest = IsolationForest(contamination=0.01, random_state=42, n_estimators=100)
iso_forest.fit(X_sample_scaled)
df['anomaly_score'] = iso_forest.score_samples(X_full_scaled)
df['is_anomaly'] = (iso_forest.predict(X_full_scaled) == -1).astype(int)

new_unsup_cols = (list(cluster_onehot.columns) + 
                  [f'cluster_{i}_dist' for i in range(n_clusters)] + 
                  ['anomaly_score', 'is_anomaly'])
feature_cols = feature_cols + [x for x in new_unsup_cols if x not in feature_cols]

In [21]:
feature_cols

['amount_ngn',
 'fee_ngn',
 'balance_after_ngn',
 'fee_rate',
 'balance_to_amount',
 'is_round_amount',
 'is_small_amount',
 'transaction_type_airtime',
 'transaction_type_billpay',
 'transaction_type_cashin',
 'transaction_type_cashout',
 'transaction_type_data',
 'transaction_type_p2p_receive',
 'transaction_type_p2p_send',
 'channel_agent',
 'channel_app',
 'channel_ussd',
 'channel_web',
 'device_os_android',
 'device_os_feature_phone',
 'device_os_ios',
 'kyc_tier_tier1',
 'kyc_tier_tier2',
 'kyc_tier_tier3',
 'wallet_txn_count',
 'wallet_total_amount',
 'wallet_avg_amount',
 'wallet_std_amount',
 'wallet_avg_fee',
 'wallet_type_diversity',
 'wallet_channel_diversity',
 'agent_txn_count',
 'agent_total_amount',
 'agent_avg_amount',
 'agent_cashout_ratio',
 'txn_count_1h',
 'amount_sum_1h',
 'amount_mean_1h',
 'amount_std_1h',
 'txn_count_24h',
 'amount_sum_24h',
 'amount_mean_24h',
 'amount_std_24h',
 'txn_count_7D',
 'amount_sum_7D',
 'amount_mean_7D',
 'amount_std_7D',
 'time_si

# 6. Нейросетевые признаки (автоэнкодер на GPU)

In [23]:
df[feature_cols].to_parquet(r'../data/df_stage1.parquet', compression='gzip',index=False)

In [26]:

df[df.select_dtypes(include=['bool']).columns] = \
    df.select_dtypes(include=['bool']).astype(int)

In [32]:
X = df[feature_cols].fillna(0).values
X_tensor = torch.tensor(X, dtype=torch.float32)
dataset = TensorDataset(X_tensor)

# Разделение на train (90%) и val (10%)
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

input_dim = X.shape[1]
encoding_dim = 20

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [36]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim, encoding_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, encoding_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim)
        )
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded

In [37]:
model = Autoencoder(input_dim, encoding_dim).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [38]:
# Early stopping parameters
patience = 3
min_delta = 1e-4
best_val_loss = float('inf')
patience_counter = 0
best_model_state = None


In [39]:
model.train()
for epoch in range(100):  # Максимальное число эпох
    # Обучение
    total_train_loss = 0
    model.train()
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} train"):
        X_batch = batch[0].to(device)
        encoded, decoded = model(X_batch)
        loss = criterion(decoded, X_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
    avg_train_loss = total_train_loss / len(train_loader)
    # Валидация
    total_val_loss = 0
    model.eval()
    with torch.no_grad():
        for batch in val_loader:
            X_batch = batch[0].to(device)
            _, decoded = model(X_batch)
            loss = criterion(decoded, X_batch)
            total_val_loss += loss.item()
    avg_val_loss = total_val_loss / len(val_loader)
    
    print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Val Loss = {avg_val_loss:.4f}")
    
    # Early stopping logic
    if avg_val_loss < best_val_loss - min_delta:
        best_val_loss = avg_val_loss
        patience_counter = 0
        best_model_state = model.state_dict().copy()
        print(f"  New best model saved with val loss {best_val_loss:.4f}")
    else:
        patience_counter += 1
        print(f"  No improvement for {patience_counter} epochs")
        if patience_counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            break


Epoch 1 train: 100%|██████████| 14063/14063 [00:36<00:00, 381.56it/s]


Epoch 1: Train Loss = 717612186.9931, Val Loss = 3934552.9554
  New best model saved with val loss 3934552.9554


Epoch 2 train: 100%|██████████| 14063/14063 [00:36<00:00, 386.73it/s]


Epoch 2: Train Loss = 6818127.5459, Val Loss = 1671598.1088
  New best model saved with val loss 1671598.1088


Epoch 3 train: 100%|██████████| 14063/14063 [00:37<00:00, 377.87it/s]


...

Epoch 8: Train Loss = 2743644.9624, Val Loss = 4734009.6745
  No improvement for 1 epochs


Epoch 9 train: 100%|██████████| 14063/14063 [00:35<00:00, 391.99it/s]


Epoch 9: Train Loss = 2594557.3718, Val Loss = 18653989.9034
  No improvement for 2 epochs


Epoch 10 train: 100%|██████████| 14063/14063 [00:36<00:00, 382.93it/s]


Epoch 10: Train Loss = 2282374.6876, Val Loss = 1355059.6833
  No improvement for 3 epochs
Early stopping triggered after 10 epochs


In [40]:
# Загружаем лучшую модель
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"Loaded best model with val loss {best_val_loss:.4f}")

Loaded best model with val loss 385686.5190


In [41]:
# Извлекаем энкодерные признаки для всего датасета
model.eval()
with torch.no_grad():
    X_tensor = X_tensor.to(device)
    encoded_features = model.encoder(X_tensor).cpu().numpy()

encoded_df = pd.DataFrame(encoded_features, columns=[f'autoenc_{i}' for i in range(encoding_dim)])
df = pd.concat([df, encoded_df], axis=1)

In [42]:
# Для удобства сохраняем датасет с признаками и метками
final_df = df.reset_index()  # возвращаем timestamp как колонку
final_df['fraud_flag'] = y_fraud
final_df['churn_30d'] = y_churn

print(f"Итоговое число признаков (без меток): {len(final_df.columns) - 2}")  # минус fraud_flag и churn_30d
final_df.to_parquet(r'../data/df.parquet', compression='gzip',index=False)

Итоговое число признаков (без меток): 109


# 7. Генерация отчёта по созданным признакам

In [44]:
# Создаём папки для отчёта, если их нет
os.makedirs("../reports/figures", exist_ok=True)

# Определяем метаданные для признаков (можно расширять)
feature_metadata = {
    # Числовые
    'amount_ngn': ('Сумма транзакции (нормализованная)', 'Объём операции, влияет на комиссию и риск'),
    'fee_ngn': ('Комиссия (нормализованная)', 'Расходы пользователя, часто завышены у мошенников'),
    'balance_after_ngn': ('Баланс после операции (нормализованный)', 'Показатель ликвидности, низкий баланс → риск'),
    'fee_rate': ('Относительная комиссия', 'Комиссия / сумма, может указывать на нестандартные операции'),
    'balance_to_amount': ('Отношение баланса к сумме', 'Показывает, насколько операция снижает баланс'),
    'is_round_amount': ('Круглая сумма', 'Мошенники часто используют круглые суммы (50k,100k,200k)'),
    'is_small_amount': ('Малая сумма (<1000 NGN)', 'Характерно для микро-транзакций, часто легитимно'),
    
    # Категориальные (one-hot)
    'transaction_type_*': ('Тип транзакции', 'Разные типы имеют разную вероятность мошенничества'),
    'channel_*': ('Канал проведения', 'USSD, app, agent, web – влияет на безопасность'),
    'device_os_*': ('ОС устройства', 'Android, iOS, feature_phone – может быть связано с уязвимостями'),
    'kyc_tier_*': ('Уровень KYC', 'Tier1 – ограниченный лимит, чаще мошенничество'),
    
    # Статистики кошелька
    'wallet_txn_count': ('Общее число транзакций кошелька', 'Активность кошелька, новые кошельки рискованнее'),
    'wallet_total_amount': ('Общая сумма операций кошелька', 'Объём, влияет на доверие'),
    'wallet_avg_amount': ('Средний чек кошелька', 'Поведенческий паттерн'),
    'wallet_std_amount': ('Стандартное отклонение сумм', 'Нестабильность может указывать на мошенничество'),
    'wallet_avg_fee': ('Средняя комиссия', 'Склонность к дорогим операциям'),
    'wallet_type_diversity': ('Разнообразие типов операций', 'Низкое разнообразие → риск оттока или мошенничества'),
    'wallet_channel_diversity': ('Разнообразие каналов', 'Мошенники часто используют один канал'),
    
    # Статистики агента
    'agent_txn_count': ('Число операций агента', 'Активность агента, новые агенты рискованны'),
    'agent_total_amount': ('Суммарный оборот агента', 'Надёжность агента'),
    'agent_avg_amount': ('Средний чек агента', 'Типичный размер выдачи/внесения'),
    'agent_cashout_ratio': ('Доля cashout у агента', 'Высокая доля выдач может указывать на отмывание'),
    
    # Временные скользящие
    'txn_count_1h': ('Число транзакций за последний час', 'Аномальная активность в коротком окне'),
    'amount_sum_1h': ('Сумма за последний час', 'Всплески объёма'),
    'amount_mean_1h': ('Средняя сумма за час', 'Изменение паттерна'),
    'amount_std_1h': ('Разброс сумм за час', 'Нестабильность'),
    'txn_count_24h': ('Число транзакций за сутки', 'Суточная активность'),
    'amount_sum_24h': ('Сумма за сутки', 'Дневной объём'),
    'amount_mean_24h': ('Средняя сумма за сутки', 'Типичный размер'),
    'amount_std_24h': ('Разброс за сутки', 'Вариативность'),
    'txn_count_7d': ('Число транзакций за 7 дней', 'Недельная активность'),
    'amount_sum_7d': ('Сумма за 7 дней', 'Недельный объём'),
    'amount_mean_7d': ('Средняя сумма за 7 дней', 'Типичный недельный чек'),
    'amount_std_7d': ('Разброс за 7 дней', 'Долгосрочная стабильность'),
    
    # Лаги времени
    'time_since_last': ('Время с последней транзакции (сек)', 'Длительные паузы могут указывать на взлом'),
    'time_since_last_same_type': ('Время с последней такой же операции', 'Необычная частота конкретного типа'),
    
    # Временные метки
    'hour': ('Час суток', 'Ночные операции более рискованны'),
    'dayofweek': ('День недели', 'Выходные/будни'),
    'is_weekend': ('Выходной', 'Мошенники активизируются в выходные'),
    'is_night': ('Ночь (22-6)', 'Аномальное время'),
    
    # Графовые
    'pair_txn_count': ('Число операций с данным агентом', 'Связность кошелёк-агент'),
    'pair_total_amount': ('Сумма операций с агентом', 'Объём взаимодействия'),
    'pair_avg_amount': ('Средний чек с агентом', 'Типичная сумма для пары'),
    'unique_agents_count': ('Число уникальных агентов', 'Ширина сети, мошенники часто ограничены'),
    
    # Обучение без учителя
    'cluster_label': ('Метка кластера', 'Группа схожих транзакций (0..9)'),
    'cluster_*_dist': ('Расстояние до центроида кластера', 'Чем дальше, тем менее типична транзакция'),
    'anomaly_score': ('Оценка аномальности (Isolation Forest)', 'Ниже – более аномально'),
    'is_anomaly': ('Бинарный флаг аномалии', 'Обнаружена выбросная транзакция'),
    
    # Автоэнкодер
    'autoenc_*': ('Латентные признаки автоэнкодера', 'Сжатое представление всех признаков, 20 компонент')
}

In [47]:
# Функция для генерации отчёта
def generate_report(df, feature_metadata, target_col='fraud_flag'):
    report_lines = []
    report_lines.append("# Отчёт по созданным признакам")
    report_lines.append(f"\nДата генерации: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    report_lines.append(f"\nКоличество строк в датасете: {len(df):,}")
    report_lines.append(f"\nЦелевая переменная: `{target_col}`")
    
    # Общая информация
    report_lines.append("\n## Общая структура признаков")
    report_lines.append(f"- Всего признаков (без меток): {len(df.columns) - 2}")
    report_lines.append(f"- Числовые признаки: {len(df.select_dtypes(include=['float64','int64']).columns)}")
    report_lines.append(f"- Категориальные: {len(df.select_dtypes(include=['object','category']).columns)}")
    
    # Список всех признаков
    report_lines.append("\n## Список признаков с описанием")
    report_lines.append("\n| Признак | Тип | Описание | Экономическая интерпретация |")
    report_lines.append("|---------|-----|----------|-----------------------------|")
    
    for col in df.columns:
        if col in ['fraud_flag', 'churn_30d', 'transaction_id', 'wallet_id', 'agent_id', 'timestamp']:
            continue
        col_type = df[col].dtype
        if col in feature_metadata:
            desc, econ = feature_metadata[col]
        else:
            # Для динамически сгенерированных (one-hot, cluster distances, autoenc)
            if col.startswith('transaction_type_'):
                desc = f"Тип транзакции: {col.replace('transaction_type_','')}"
                econ = "Индикатор типа операции"
            elif col.startswith('channel_'):
                desc = f"Канал: {col.replace('channel_','')}"
                econ = "Индикатор канала"
            elif col.startswith('device_os_'):
                desc = f"ОС: {col.replace('device_os_','')}"
                econ = "Индикатор ОС"
            elif col.startswith('kyc_tier_'):
                desc = f"KYC: {col.replace('kyc_tier_','')}"
                econ = "Индикатор уровня KYC"
            elif col.startswith('cluster_') and '_dist' in col:
                desc = f"Расстояние до кластера {col.split('_')[1]}"
                econ = "Близость к типичным транзакциям кластера"
            elif col.startswith('autoenc_'):
                desc = f"Латентный признак автоэнкодера {col.split('_')[1]}"
                econ = "Нелинейная комбинация исходных признаков"
            else:
                desc = "Автоматически сгенерированный признак"
                econ = "Не указано"
        report_lines.append(f"| {col} | {col_type} | {desc} | {econ} |")
    
    # Статистики для числовых признаков
    report_lines.append("\n## Статистики числовых признаков")
    numeric_cols = df.select_dtypes(include=['float64','int64']).columns
    numeric_stats = df[numeric_cols].describe().T
    report_lines.append("\n```")
    report_lines.append(numeric_stats.to_string())
    report_lines.append("```")
    
    # Распределения категориальных признаков (топ-5)
    cat_cols = df.select_dtypes(include=['object','category']).columns
    if len(cat_cols) > 0:
        report_lines.append("\n## Распределение категориальных признаков (топ-5 значений)")
        for col in cat_cols:
            if col in ['fraud_flag', 'churn_30d']:
                continue
            report_lines.append(f"\n### {col}")
            val_counts = df[col].value_counts().head(5)
            report_lines.append("```")
            report_lines.append(val_counts.to_string())
            report_lines.append("```")
    
    # Визуализации
    report_lines.append("\n## Визуализации")
    # 1. Гистограммы для 20 наиболее важных числовых признаков (по размаху)
    top_numeric = numeric_cols[:20]  # ограничим для отчёта
    for col in top_numeric:
        plt.figure(figsize=(8,4))
        sns.histplot(df[col], bins=50, kde=True)
        plt.title(f'Distribution of {col}')
        plt.xlabel(col)
        plt.ylabel('Frequency')
        plot_path = f"../reports/figures/{col}_hist.png"
        plt.savefig(plot_path, bbox_inches='tight')
        plt.close()
        report_lines.append(f"\n![{col} distribution](figures/{col}_hist.png)")
    
    # 2. Корреляционная матрица с целевой (fraud_flag)
    if target_col in df.columns:
        plt.figure(figsize=(12,10))
        # Строим корреляцию только числовых признаков + целевой
        numeric_cols = df.select_dtypes(include=['float64','int64']).columns.tolist()
        if target_col not in numeric_cols:
            numeric_cols.append(target_col)
        corr = df[numeric_cols].corr()
        sns.heatmap(corr, annot=False, cmap='coolwarm', center=0)
        plt.title(f'Correlation matrix')
        plot_path = f"../reports/figures/correlation_matrix.png"
        plt.savefig(plot_path, bbox_inches='tight')
        plt.close()
        report_lines.append(f"\n![Correlation matrix](figures/correlation_matrix.png)")
        
        # Топ-10 признаков по корреляции с target (исключая саму целевую)
        corr_series = corr[target_col].drop(target_col).abs().sort_values(ascending=False)
        corr_target_top10 = corr_series.head(10)
        report_lines.append("\n### Топ-10 признаков по абсолютной корреляции с fraud_flag")
        report_lines.append("```")
        report_lines.append(corr_target_top10.to_string())
        report_lines.append("```")
    
    # Сохраняем отчёт
    with open("../reports/feature_report.md", "w", encoding="utf-8") as f:
        f.write("\n".join(report_lines))
    
    print("Отчёт сохранён в ../reports/feature_report.md")
    print("Графики сохранены в ../reports/figures/")

In [48]:
# Генерируем отчёт на основе final_df
# Перед вызовом убедимся, что все необходимые колонки есть
generate_report(final_df, feature_metadata, target_col='fraud_flag')

C:\Users\Admin\AppData\Local\Temp\ipykernel_22236\91404811.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  report_lines.append(f"- Категориальные: {len(df.select_dtypes(include=['object','category']).columns)}")
C:\Users\Admin\AppData\Local\Temp\ipykernel_22236\91404811.py:60: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning

Отчёт сохранён в ../reports/feature_report.md
Графики сохранены в ../reports/figures/
